# 3I — Incremental Stream: Weekly Tabular Feature Engineering

## Objective

Prepare the national weekly time series for the four CPU models in Stream I:

- `SGDRegressor(loss='squared_error')`
- `SGDRegressor(loss='huber')`
- `SGDRegressor(loss='epsilon_insensitive')`
- `MLPRegressor`

This notebook performs only deterministic, past-only feature engineering. It does **not** fit feature or target scalers. Each model notebook fits its scalers separately inside each Optuna training fold, then refits them on the complete 80% training period. This avoids validation leakage.

A row represents a forecast made at the end of an observed week for the **next week**. The target is the next week's national accident count.

## 0. Load the common weekly series

In [ ]:
from pathlib import Path
import os, json, warnings, gc, time, copy, random, sys
import numpy as np
import pandas as pd

SEED = 34
random.seed(SEED)
np.random.seed(SEED)

ON_KAGGLE = Path('/kaggle/working').exists()
PROJECT_ROOT = Path('/kaggle/working/us_accidents_weekly_forecasting') if ON_KAGGLE else Path('./us_accidents_weekly_forecasting')
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
MODEL_DIR = PROJECT_ROOT / 'models'
REPORT_DIR = PROJECT_ROOT / 'reports'
for _p in (PROJECT_ROOT, ARTIFACT_DIR, MODEL_DIR, REPORT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

# Set this only when more than one prior notebook output is attached as a Kaggle input.
ARTIFACT_INPUT_ROOT = None  # e.g. Path('/kaggle/input/us-accidents-weekly-stage-2')

def resolve_artifact(filename, required=True):
    candidates = []
    if ARTIFACT_INPUT_ROOT is not None:
        p = Path(ARTIFACT_INPUT_ROOT) / filename
        if p.exists():
            candidates.append(p)
    p = ARTIFACT_DIR / filename
    if p.exists():
        candidates.append(p)
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates.extend(kaggle_input.rglob(filename))
    # Deduplicate resolved paths and prefer the most recently modified attached artifact.
    unique = {str(p.resolve()): p for p in candidates}
    candidates = list(unique.values())
    if not candidates:
        if required:
            raise FileNotFoundError(
                f'Could not locate {filename}. Add the preceding notebook output as a Kaggle input, '
                f'or set ARTIFACT_INPUT_ROOT. Local output searched: {ARTIFACT_DIR}'
            )
        return None
    chosen = max(candidates, key=lambda p: p.stat().st_mtime)
    if len(candidates) > 1:
        print(f'Found {len(candidates)} copies of {filename}; using newest: {chosen}')
    return chosen

print('Project root:', PROJECT_ROOT.resolve())
print('Seed:', SEED)

from pandas.api.types import is_numeric_dtype

national_path = resolve_artifact('weekly_national_counts.parquet')
split_path = resolve_artifact('split_metadata.json')
national = pd.read_parquet(national_path).sort_values('week_id').reset_index(drop=True)
split_meta = json.loads(split_path.read_text())
TRAIN_END_WEEK_ID = int(split_meta['train_end_week_id'])

required = {'week_id','week_start','national_accident_count'}
if not required.issubset(national.columns):
    raise ValueError(f'Weekly national data must contain {sorted(required)}')
national['week_start'] = pd.to_datetime(national['week_start'])
counts = national['national_accident_count'].to_numpy(dtype=np.float64)
week_ids = national['week_id'].to_numpy(dtype=np.int32)
week_dates = national['week_start'].to_numpy()
if not np.array_equal(week_ids, np.arange(1, len(week_ids)+1)):
    raise AssertionError('week_id must be sequential, beginning at 1.')
print('Weeks:', len(counts), '| train end:', TRAIN_END_WEEK_ID)

## 1. Build strictly past-only one-step-ahead features

For target week `k`, the forecast origin is week `k-1`. Every lag and rolling statistic below uses values no later than that origin. The 52-week lag means the first supervised target is week 53.

In [ ]:
LAGS = [1, 2, 3, 4, 8, 13, 26, 52]
ROLLING_WINDOWS = [4, 13, 26, 52]

feature_rows = []
target_values = []
metadata_rows = []

for target_idx in range(max(LAGS), len(counts)):
    # target_idx is zero-based; its target week_id is target_idx + 1.
    origin_idx = target_idx - 1
    row = {}
    for lag in LAGS:
        value = counts[origin_idx - (lag - 1)]
        row[f'log_count_lag_{lag}'] = float(np.log1p(max(value, 0.0)))
    for window in ROLLING_WINDOWS:
        hist = counts[origin_idx - window + 1: origin_idx + 1]
        row[f'log_rolling_mean_{window}'] = float(np.log1p(max(float(np.mean(hist)), 0.0)))
        row[f'log_rolling_std_{window}'] = float(np.log1p(max(float(np.std(hist, ddof=0)), 0.0)))
    if origin_idx >= 1:
        previous = counts[origin_idx - 1]
        row['relative_change_1w'] = float((counts[origin_idx] - previous) / max(abs(previous), 1.0))
    else:
        row['relative_change_1w'] = 0.0

    target_date = pd.Timestamp(week_dates[target_idx])
    iso_week = int(target_date.isocalendar().week)
    row['target_week_sin'] = float(np.sin(2.0 * np.pi * iso_week / 52.1775))
    row['target_week_cos'] = float(np.cos(2.0 * np.pi * iso_week / 52.1775))
    row['target_week_id'] = float(week_ids[target_idx])
    row['target_year'] = float(target_date.year)
    row['target_quarter_sin'] = float(np.sin(2.0 * np.pi * target_date.quarter / 4.0))
    row['target_quarter_cos'] = float(np.cos(2.0 * np.pi * target_date.quarter / 4.0))

    feature_rows.append(row)
    target_values.append(float(counts[target_idx]))
    metadata_rows.append({
        'target_week_id': int(week_ids[target_idx]),
        'forecast_origin_week_id': int(week_ids[origin_idx]),
        'target_week_start': target_date,
        'actual_count': float(counts[target_idx]),
        'week_days_within_source_bounds': int(national.iloc[target_idx]['week_days_within_source_bounds']),
    })

X_all = pd.DataFrame(feature_rows).replace([np.inf, -np.inf], np.nan)
if X_all.isna().any().any():
    raise ValueError('Unexpected missing values in deterministic lag/calendar features.')
y_all = np.asarray(target_values, dtype=np.float64)
meta_all = pd.DataFrame(metadata_rows)
FEATURE_NAMES = X_all.columns.tolist()

train_mask = meta_all.target_week_id.to_numpy() <= TRAIN_END_WEEK_ID
test_mask = ~train_mask
X_train_raw = X_all.loc[train_mask].to_numpy(dtype=np.float64)
y_train_raw = y_all[train_mask]
X_test_raw = X_all.loc[test_mask].to_numpy(dtype=np.float64)
y_test_raw = y_all[test_mask]
train_meta = meta_all.loc[train_mask].reset_index(drop=True)
test_meta = meta_all.loc[test_mask].reset_index(drop=True)

if len(X_train_raw) < 100 or len(X_test_raw) < 10:
    raise ValueError('Not enough supervised rows after creating the 52-week lag.')
if train_meta.target_week_id.max() >= test_meta.target_week_id.min():
    raise AssertionError('Train/test target weeks overlap or are out of order.')
print('Feature matrix:', X_train_raw.shape, '| test:', X_test_raw.shape)
print('Features:', FEATURE_NAMES)

## 2. Persist raw feature matrices and metadata

The model notebooks fit `StandardScaler` and the `log1p` target scaler inside every temporal CV fold. Fitting those scalers here on the entire training period would leak later validation-fold distribution information into earlier folds.

In [ ]:
np.save(ARTIFACT_DIR / 'stream_i_X_train_raw.npy', X_train_raw.astype(np.float32))
np.save(ARTIFACT_DIR / 'stream_i_y_train_raw.npy', y_train_raw.astype(np.float32))
np.save(ARTIFACT_DIR / 'stream_i_X_test_raw.npy', X_test_raw.astype(np.float32))
np.save(ARTIFACT_DIR / 'stream_i_y_test_raw.npy', y_test_raw.astype(np.float32))
train_meta.to_parquet(ARTIFACT_DIR / 'stream_i_train_metadata.parquet', index=False)
test_meta.to_parquet(ARTIFACT_DIR / 'stream_i_test_metadata.parquet', index=False)

feature_map = {
    'target': 'national_accident_count in target_week_id',
    'forecast_horizon': 1,
    'lags': LAGS,
    'rolling_windows': ROLLING_WINDOWS,
    'features': FEATURE_NAMES,
    'feature_transform': 'count lags and rolling statistics use log1p; relative change is raw ratio; calendar variables are deterministic',
    'leakage_rule': 'all count-derived features for target week k use observations only through week k-1',
    'scaling': 'not performed here; each 4I notebook fits feature/target scalers within each CV training fold',
    'split': split_meta,
}
(ARTIFACT_DIR / 'stream_i_feature_map.json').write_text(json.dumps(feature_map, indent=2))
print('Saved Stream I train/test arrays and feature map.')

## 3. Sanity checks: inspect selected features and the forecast origin

In [ ]:
preview = train_meta[['forecast_origin_week_id','target_week_id','target_week_start','actual_count']].head(8)
display(preview)
assert np.all(train_meta.forecast_origin_week_id.to_numpy() + 1 == train_meta.target_week_id.to_numpy())
assert np.isfinite(X_train_raw).all() and np.isfinite(X_test_raw).all()
assert np.all(y_train_raw >= 0) and np.all(y_test_raw >= 0)
print('Stream I preprocessing checks passed.')